> **Repository execution note.** Start JupyterLab from the **repository root** and place authorised source data in that same root directory (see [README](../README.md) and [Data requirements](../data/README.md)). These notebooks were originally distributed as standalone packages: any older instructions referring to an included ZIP, a `test/` folder, or bundled data do not apply to this GitHub repository. The notebooks preserve their original input schemas and experiment configurations. Results cached in the original files are intentionally excluded from this public-ready copy.

# Notebook 06 | Spatial Profiles and the Predictive Value of Tourism Accessibility
### Luís Silvano Project · *Knowledge-Based Systems* · 2018–2024

**Scientific question**  
*Do tourism accessibility profiles provide incremental explanatory and predictive value for hotel labour productivity beyond structural characteristics, regional heterogeneity, and temporal dynamics?*

**What this notebook does:** joins the spatial indicators created in Notebook 05 to the hotel panel, describes productivity by spatial profile over 2018–2024, and compares **three information sets under identical temporal splits**.

| Configuration | Inputs supplied to the model | Research question |
|---|---|---|
| **A — Structural + temporal** | Previous-year structure, region, forecast year, previous-year productivity | How well do models perform without POIs? |
| **B — A + accessibility** | A + three accessibility indices and restaurant/café density | How much do continuous spatial characteristics add? |
| **C — B + spatial cluster** | B + cluster **recomputed using training data only** | Does grouping improve the *representation* of spatial features? |

**Conceptual caution:** a cluster is a function of the spatial variables from which it was derived. **C does not add new data to B.** A gain for C over B suggests a possible benefit in representation/inductive bias, not a new independent information source.

**What this is NOT:** causal evidence that tourist attractions increase productivity, or genuinely historical forecasting if the POIs were only observed in 2026. Geographic indicators are treated as a spatial *snapshot* with a date still to be confirmed. The entire 2024 evaluation is a *retrospective audit*, because that year was examined in earlier notebooks.


## 0. Quick guide for Luís — corrected version

**This notebook can work directly with the `resultados_espaciais_05` folder generated by Notebook 05.** There is no need to rename or manually move the CSV files.

### Recommended folder structure

```text
test/
├── KBS_Luis_Silvano_06_CORRIGIDO_Leitura_Automatica.ipynb
└── resultados_espaciais_05/
    ├── 09_indicadores_e_clusters_por_hotel.csv
    └── 10_painel_com_indicadores_espaciais.csv
```

1. Place **this notebook in the `test` folder**, next to the `resultados_espaciais_05` subfolder. Open it from `test` in Jupyter and select **Kernel → Restart & Run All**.
2. The corrected version first looks for `10_painel_com_indicadores_espaciais.csv` (annual panel) and `09_indicadores_e_clusters_por_hotel.csv` (one row per hotel). If unavailable, it tries the older files `hoteis_2018_2024_subset.csv` and `indicadores_espaciais_05.csv`.
3. Indicators already present in the enriched panel are **replaced in a controlled manner** with those from `09_...`; the **descriptive cluster from Notebook 05 is not used to train predictive models**. A new K-Means model is fitted within each training fold, without using productivity.
4. If the updated files contain five-neighbour (`k=5`) indicators for all three categories, the notebook uses them. If one category has insufficient coverage, it consistently uses `k=1` and issues a warning. **It does not fabricate `k=5` values.**
5. For modern models, optionally install `%pip install "pytabkit[models]"`. Unavailable models are logged in `resultados_KBS_06/17_falhas_modelos.csv`.
6. Inspect the results, figures, and merge report in `resultados_KBS_06`. The 2024 evaluation remains **a retrospective audit**, not a blind test.

**Core dependencies, if needed:**

```python
%pip install numpy pandas matplotlib seaborn scipy scikit-learn xgboost lightgbm catboost
```

> **Note:** if you launch Jupyter from another working directory, the configuration cell displays `ROOT`. Place the notebook in the directory shown above or change only `ROOT` to point to `test`.


## 1. Study configuration — parameters that may be changed


In [ ]:
from pathlib import Path
import os, sys, time, json, hashlib, importlib.util, traceback, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

SEMENTE = 42
THREADS = 2
# 'completo' requests all models, including optional ones; 'verificacao' runs the core set.
PERFIL = os.getenv('KBS06_PERFIL', 'completo')
MODELOS_RAPIDOS = ['ridge', 'random_forest', 'xgboost', 'lightgbm']
MODELOS_COMPLETOS = MODELOS_RAPIDOS + ['catboost', 'tabm', 'realmlp']
MODELOS = MODELOS_COMPLETOS if PERFIL == 'completo' else MODELOS_RAPIDOS

ANOS_CV = [2021, 2022, 2023]
ANO_AUDITORIA = 2024
EXECUTAR_AUDITORIA_2024 = True
K_CLUSTERS = 5                 # prespecified using Notebook 05
POLITICA_Y_TREINO = 'p995'    # 'original' or 'p995': clip the training target only
N_BOOT = 250
EXECUTAR_SENSIBILIDADE_2020 = True
MODELOS_SENS = ['ridge','xgboost']
SENSIBILIDADE_FINANCEIRA = False  # optional: include employee counts/revenue from t-1
ROOT = Path.cwd()
OUT = ROOT / 'resultados_KBS_06'
OUT.mkdir(parents=True, exist_ok=True)

CORES_PERFIS = {1:'#0072B2',2:'#D55E00',3:'#009E73',4:'#CC79A7',5:'#E69F00',6:'#56B4E9'}
CORES_CENARIOS = {'A_estrutura_tempo':'#6C757D', 'B_mais_espaco':'#0072B2',
                  'C_mais_cluster':'#009E73'}
plt.rcParams.update({'figure.dpi':115, 'savefig.dpi':220,
                     'axes.titlesize':12, 'axes.labelsize':10,'font.size':10})
np.random.seed(SEMENTE)
print('Profile:', PERFIL, '| Models:', MODELOS)
print('Working folder:',ROOT.resolve(), '| Outputs:',OUT.resolve())

## 2. Data provenance and dataset joining — corrected

**Preferred option: use the two files exported by Notebook 05**, without reconstructing the panel:

- `resultados_espaciais_05/10_painel_com_indicadores_espaciais.csv`: **hotel × year** table already enriched by Notebook 05; contains the target `f_labprodct`, structural characteristics, and identifiers.
- `resultados_espaciais_05/09_indicadores_e_clusters_por_hotel.csv`: table with **one row per hotel**, accessibility indices, and the exploratory cluster.

The code also supports the two older CSV files in the main folder. `hotel_key` is checked against `ht_registration_number` to avoid joining different hotels incorrectly. Spatial indicators already present in the panel are removed before the merge to avoid `_x`/`_y` columns.

**Rule for predictive experiments:** clusters exported by Notebook 05 are **not** used as inputs; the clusters in C are reconstructed within each temporal split using training data only. The exported cluster identifier is used **only** for descriptive plots covering 2018–2024.


In [ ]:
def normalize_hotel_id(valor):
    'Normalise the hotel registration identifier without confusing 3317 and 3317.0.'
    if pd.isna(valor):
        return None
    try:
        v = float(valor)
        if np.isfinite(v) and v.is_integer():
            return str(int(v))
    except (TypeError, ValueError, OverflowError):
        pass
    return str(valor).strip() or None

# The Jupyter working folder is usually the notebook's folder.
PASTA_05 = ROOT / 'resultados_espaciais_05'

# Prefer the LATEST complete files generated by Notebook 05.
CANDIDATOS_PAINEL = [
    PASTA_05 / '10_painel_com_indicadores_espaciais.csv',
    ROOT / '10_painel_com_indicadores_espaciais.csv',
    ROOT / 'hoteis_2018_2024_subset.csv',
]
ARQ_PAINEL = next((p for p in CANDIDATOS_PAINEL if p.is_file()), None)

CANDIDATOS_ESPACO = [
    (ARQ_PAINEL.parent / '09_indicadores_e_clusters_por_hotel.csv') if ARQ_PAINEL else PASTA_05 / '09_indicadores_e_clusters_por_hotel.csv',
    PASTA_05 / '09_indicadores_e_clusters_por_hotel.csv',
    ROOT / '09_indicadores_e_clusters_por_hotel.csv',
    ROOT / 'indicadores_espaciais_05.csv',
]
ARQ_ESPACO = next((p for p in CANDIDATOS_ESPACO if p.is_file()), None)

if ARQ_PAINEL is None or ARQ_ESPACO is None:
    print('Working directory:', ROOT.resolve())
    print('Candidate panel files:', *[str(p) for p in CANDIDATOS_PAINEL], sep='\n  - ')
    print('Candidate spatial files:', *[str(p) for p in CANDIDATOS_ESPACO], sep='\n  - ')
    raise FileNotFoundError(
        'Required CSV files not found. Open the notebook in the test folder, '
        'next to the resultados_espaciais_05 subfolder containing files 09 and 10.'
    )

painel = pd.read_csv(ARQ_PAINEL, low_memory=False)
espaco = pd.read_csv(ARQ_ESPACO, low_memory=False)
print('Panel loaded:', ARQ_PAINEL.resolve(), '| formato:', painel.shape)
print('Spatial indicators loaded:', ARQ_ESPACO.resolve(), '| formato:', espaco.shape)

obrigatorias = ['ht_registration_number', 'ht_hotel_chain_d', 'f_year', 'f_labprodct',
                'ht_capacity', 'ht_quality_stars']
faltas = [c for c in obrigatorias if c not in painel.columns]
if faltas:
    raise ValueError(f'The annual panel does not contain {faltas}. Available columns: {list(painel.columns)[:40]}')
if 'hotel_key' not in espaco.columns:
    raise ValueError('Spatial file 09 must contain a hotel_key column.')

# Preserve, verify, and normalise the hotel registration identifier.
chave_calculada = painel['ht_registration_number'].map(normalize_hotel_id)
if 'hotel_key' in painel.columns:
    chave_exportada = painel['hotel_key'].map(normalize_hotel_id)
    conflitos = chave_calculada.notna() & chave_exportada.notna() & chave_calculada.ne(chave_exportada)
    if conflitos.any():
        raise ValueError(
            f'The panel has {int(conflitos.sum())} hotel_key values inconsistent with '
            'ht_registration_number. Verify identifiers before merging.'
        )
painel['hotel_key'] = chave_calculada
espaco['hotel_key'] = espaco['hotel_key'].map(normalize_hotel_id)
espaco = espaco.dropna(subset=['hotel_key']).copy()
if espaco.hotel_key.duplicated().any():
    exemplos = espaco.loc[espaco.hotel_key.duplicated(False), 'hotel_key'].unique()[:5].tolist()
    raise ValueError(f'The spatial file contains duplicate hotel_key values (e.g.: {exemplos}).')

painel['f_year'] = pd.to_numeric(painel['f_year'], errors='coerce')
painel['ht_hotel_chain_d'] = pd.to_numeric(painel['ht_hotel_chain_d'], errors='coerce')
painel['f_labprodct'] = pd.to_numeric(painel['f_labprodct'], errors='coerce')
painel = painel[painel.f_year.between(2018, 2024) & painel.ht_hotel_chain_d.eq(0)].copy()
painel = painel.dropna(subset=['hotel_key'])
painel['f_year'] = painel['f_year'].astype(int)
if painel.duplicated(['hotel_key', 'f_year']).any():
    exemplos = painel.loc[painel.duplicated(['hotel_key', 'f_year'], False),
                          ['hotel_key', 'f_year']].head(5).to_dict('records')
    raise ValueError(f'Duplicate hotel–year observations in the panel. Examples: {exemplos}')

K5_CANDIDATAS = ['a_castelos_k5', 'a_igrejas_k5', 'a_praias_k5']
K1_CANDIDATAS = ['a_castelos_k1', 'a_igrejas_k1', 'a_praias_k1']
cobertura_k5 = {c: round(pd.to_numeric(espaco[c], errors='coerce').notna().mean(), 3)
                if c in espaco else 0.0 for c in K5_CANDIDATAS}
completo_k5 = all(c in espaco.columns and pd.to_numeric(espaco[c], errors='coerce').notna().mean() >= .95
                  for c in K5_CANDIDATAS)
K_VIZINHOS = 5 if completo_k5 else 1
ESP_COLS = [f'a_castelos_k{K_VIZINHOS}', f'a_igrejas_k{K_VIZINHOS}',
            f'a_praias_k{K_VIZINHOS}', 'log_dens_restaurantes_r0p8km']
faltas_esp = [c for c in ESP_COLS if c not in espaco.columns]
if faltas_esp:
    raise ValueError(f'Spatial indicators {faltas_esp} are missing from file 09. '
                     'First run the full export from Notebook 05.')
for c in ESP_COLS:
    espaco[c] = pd.to_numeric(espaco[c], errors='coerce')

# Panel file 10 may already contain the cluster and four indices; this must NOT cause
# _x/_y suffixes or introduce global cluster labels into training.
colunas_anteriores = [c for c in ESP_COLS + ['cluster', 'perfil_nome', 'n_dimensoes_validas']
                     if c in painel.columns]
if colunas_anteriores:
    print('Removed from panel before merging (already exported by Notebook 05):', colunas_anteriores)
painel_sem_espaco = painel.drop(columns=colunas_anteriores, errors='ignore')

# Only the original spatial indicators from file 09 are merged here.
base = painel_sem_espaco.merge(espaco[['hotel_key'] + ESP_COLS],
                              on='hotel_key', how='inner', validate='many_to_one')
base = base.replace([np.inf, -np.inf], np.nan)
base['n_esp_validos'] = base[ESP_COLS].notna().sum(axis=1)
base = base[base.n_esp_validos >= 3].copy()

print('Independent hotels in panel:', painel.hotel_key.nunique())
print('Hotels with spatial indicators:', espaco.hotel_key.nunique())
print('Selected neighbour count k:', K_VIZINHOS, '| cobertura k=5:', cobertura_k5)
print('Merge complete:', len(base), 'observations,', base.hotel_key.nunique(), 'hotels')
if K_VIZINHOS == 1:
    print('WARNING: k=5 does not reach ≥95% completeness across all three categories; k=1 was used consistently.')
if base.empty:
    raise ValueError('No observations with at least three indicators remain after merging.')
assert not any(f'{c}_x' in base.columns or f'{c}_y' in base.columns for c in ESP_COLS)

# Transparent audit of losses during merging and panel selection.
audit = pd.DataFrame([
    {'categoria': 'Independent-hotel observations in the annual dataset', 'n': len(painel)},
    {'categoria': 'Independent hotels in the annual dataset', 'n': painel.hotel_key.nunique()},
    {'categoria': 'Hotels with indicators in ≥3 dimensions', 'n': base.hotel_key.nunique()},
    {'categoria': 'Observations after spatial merging', 'n': len(base)},
    {'categoria': 'Observations with an available target', 'n': base.f_labprodct.notna().sum()},
])
display(audit)
audit.to_csv(OUT / '01_auditoria_juncao.csv', index=False)


## 3. Mathematics of indicators and spatial profiles

For hotel $i$, tourism category $c$, and the $j$th nearest point of interest, let $d^{(c)}_{i,(j)}$ denote geodesic distance (in km). The accessibility index used in Notebook 05 is:

\[
 A_i^{(c)}(k)=\frac{1}{k}\sum_{j=1}^{k}\frac{1}{1+d^{(c)}_{i,(j)}/d_0},\qquad d_0=1\,\mathrm{km}.
\]

The density of restaurants/cafés within an 800-metre radius is:

\[
D_i(0.8)=\frac{N_i(0.8)}{\pi (0.8)^2}\quad \big[\mathrm{establishments/km^2}\big],\qquad R_i=\log(1+D_i(0.8)).
\]

Combining four indicators $\mathbf{s}_i=[A_i^{\mathrm{castelos}},A_i^{\mathrm{igrejas}},A_i^{\mathrm{praias}},R_i]^\top$, K-Means learns the partition

\[
\min_{\{\mathcal C_g,\boldsymbol\mu_g\}_{g=1}^K}
\sum_{g=1}^{K}\sum_{i\in\mathcal C_g}\left\|\tilde{\mathbf s}_i-\boldsymbol\mu_g\right\|_2^2,
\qquad K=5,
\]

where $\tilde{\mathbf s}_i$ results from **imputation, percentile clipping, and standardisation estimated on training data only**. The same transformations and centroids are applied in validation years, without refitting.

The one-year-ahead prediction is $\hat y_{i,t}=f(\mathbf x_{i,t-1}, y_{i,t-1},\mathbf s_i,\mathbb 1[g_i=g])$. Configuration A excludes $\mathbf s_i$ and $g_i$; B includes $\mathbf s_i$; C also includes the cluster. **No year-$t$ feature derived from revenue or the number of employees is included among the main predictors.**


## 4. Descriptive characterisation — productivity by profile from 2018 to 2024


In [ ]:
# Descriptive profiles come from Notebook 05 and are not used for predictive training.
if 'cluster' not in espaco.columns:
    print('WARNING: exported data have no descriptive clusters. The annual plot will be skipped.')
    descritivo = pd.DataFrame()
else:
    descritivo = base[['hotel_key','f_year','f_labprodct']].merge(
        espaco[['hotel_key','cluster']].drop_duplicates('hotel_key'),on='hotel_key',validate='many_to_one')
    descritivo['cluster']=pd.to_numeric(descritivo['cluster'],errors='coerce')
    descritivo=descritivo.dropna(subset=['cluster','f_labprodct'])
    descritivo=descritivo[descritivo.f_labprodct.ge(0)].copy()
    descritivo['cluster']=descritivo['cluster'].astype(int)
    resumo_cluster_ano=(descritivo.groupby(['cluster','f_year'])['f_labprodct']
       .agg(n='size',mediana='median',media='mean',q25=lambda x:x.quantile(.25),
            q75=lambda x:x.quantile(.75)).reset_index())
    display(resumo_cluster_ano.head(15).round(2))
    resumo_cluster_ano.to_csv(OUT/'02_produtividade_anual_cluster.csv',index=False)

    fig,axs=plt.subplots(1,2,figsize=(14,5.2))
    for grupo,g in resumo_cluster_ano.groupby('cluster'):
        g=g.sort_values('f_year')
        c=CORES_PERFIS.get(grupo,'#333333')
        axs[0].plot(g.f_year,g.mediana,marker='o',lw=1.9,color=c,label=f'Cluster {grupo}')
        axs[0].fill_between(g.f_year,g.q25,g.q75,color=c,alpha=.10)
        axs[1].plot(g.f_year,g.n,marker='s',lw=1.6,color=c,label=f'Cluster {grupo}')
    for ax in axs:
        ax.axvspan(2019.6,2020.4,color='#F4C36A',alpha=.27,label='2020: pandemic shock')
        ax.set_xticks(list(range(2018,2025)))
        ax.set_xlabel('Year')
    axs[0].set(ylabel='Labour productivity (median and interquartile range)',
               title='Trends by spatial profile | observed values')
    axs[1].set(ylabel='Observations with valid targets',title='Sample coverage by profile and year')
    axs[0].legend(ncol=2,fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT/'figura_01_evolucao_temporal_clusters.png',bbox_inches='tight')
    plt.show()

### 4.1. The 2019 → 2020 shock in **the same hotel cohort**

A year's mean or median may change because the set of observed firms changes. To control **only for sample composition**, this analysis uses hotels with observed productivity **in both years**. The difference $\Delta_i=y_{i,2020}-y_{i,2019}$ is descriptive, not causal; spatial groups represent the geography recorded in Notebook 05, which may reflect a later *snapshot*.


In [ ]:
if not descritivo.empty:
    dupla=descritivo[descritivo.f_year.isin([2019,2020])].pivot_table(
        index=['hotel_key','cluster'],columns='f_year',values='f_labprodct',aggfunc='first')
    dupla=dupla.dropna(subset=[2019,2020]).reset_index()
    dupla['variacao']=dupla[2020]-dupla[2019]
    dupla['queda']=(dupla['variacao']<0)
    choque=dupla.groupby('cluster').agg(
        hoteis=('hotel_key','nunique'),variacao_mediana=('variacao','median'),
        variacao_media=('variacao','mean'),fracao_com_queda=('queda','mean')).reset_index()
    display(choque.round(3))
    choque.to_csv(OUT/'03_choque_pandemia_por_cluster.csv',index=False)
    fig,axs=plt.subplots(1,2,figsize=(12.5,4.5))
    cols=[CORES_PERFIS.get(int(k),'grey') for k in choque.cluster]
    axs[0].bar(choque.cluster.astype(str),choque.variacao_mediana,color=cols)
    axs[0].axhline(0,color='black',lw=.8)
    axs[0].set(title='Change from 2019 to 2020 | same hotels',xlabel='Cluster',ylabel='Δ median productivity')
    axs[1].bar(choque.cluster.astype(str),100*choque.fracao_com_queda,color=cols)
    axs[1].set(ylim=(0,100),title='Proportion declining from 2019 to 2020',
               xlabel='Cluster',ylabel='Hotels with a decline (%)')
    fig.tight_layout()
    fig.savefig(OUT/'figura_02_pandemia_pares_por_cluster.png',bbox_inches='tight')
    plt.show()

## 5. Constructing the prediction target: only information available up to $t-1$

A `lag1` is the value from **the previous year** for the same hotel. We construct structural features for year $t-1$ using `groupby` and `shift(1)`, retaining **only consecutive-year pairs**. If there is no observation at $t-1$, we do not treat a $t-2$ value as though it came from $t-1$.

**Common cohort:** experiments A/B/C use exactly the same hotel–year rows. A performance gain therefore cannot be attributed to an arbitrary reduction in sample size in the more sophisticated scenario.


In [ ]:
COL_Y='f_labprodct'
COL_ANO='f_year'
ESTRUTURA=['ht_capacity','ht_quality_stars','ht_number_of_accommodation_units',
           'ht_spa_d','ht_gym_d','ht_outdoor_pools_d','ht_meeting_rooms_d']
FIN_HISTORICO=['f_number_of_employees','f_operating_revenue_turnover2']
USAR_LAGS=ESTRUTURA+(FIN_HISTORICO if SENSIBILIDADE_FINANCEIRA else [])
for c in USAR_LAGS:
    if c not in base.columns: raise ValueError('Missing structural column: '+c)
    base[c]=pd.to_numeric(base[c],errors='coerce')
base=base.sort_values(['hotel_key',COL_ANO]).copy()
grupo=base.groupby('hotel_key',sort=False)
base['ano_lag1']=grupo[COL_ANO].shift(1)
base['prod_lag1']=grupo[COL_Y].shift(1)
base['regiao_lag1']=grupo['ht_nuts_2_newcriteria'].shift(1)
for c in USAR_LAGS:
    base[c+'_lag1']=grupo[c].shift(1)

PARES = base[(base[COL_ANO]-base.ano_lag1).eq(1) &
             base[COL_Y].notna() & base.prod_lag1.notna() &
             base[COL_Y].ge(0) & base.prod_lag1.ge(0)].copy()
PARES['regiao_lag1']=PARES['regiao_lag1'].fillna('Region not specified').astype(str)
FEATS_A=['prod_lag1','f_year']+[c+'_lag1' for c in USAR_LAGS]
FEATS_B=FEATS_A+ESP_COLS
CONFIGURACOES=['A_estrutura_tempo','B_mais_espaco','C_mais_cluster']
print('Consecutive-year pairs with valid productivity:',len(PARES), '| Hotels:',PARES.hotel_key.nunique())
contagens=PARES.groupby('f_year').agg(n=('hotel_key','size'),hoteis=('hotel_key','nunique'))
display(contagens)
contagens.to_csv(OUT/'04_pares_modelagem_por_ano.csv')
assert PARES.f_year.min()>=2019
assert PARES.groupby(['hotel_key','f_year']).size().max()==1
assert not any(c in FEATS_B for c in ['f_number_of_employees','f_operating_revenue_turnover2'])

## 6. Walk-forward protocol and prevention of information leakage

Forecast origins are **2021**, **2022**, and **2023**. To predict year $v$, training uses only target years $t<v$; target values from the evaluation year never enter model fitting. The **2024 audit is separate**.

For model C, K-Means is fitted on **unique hotels present in the training set**, preventing hotels observed in more years from receiving greater weight when defining centroids. Validation hotels are assigned to the existing centroids only. Spatial imputation, p01/p99 clipping, and `StandardScaler` are also fitted using training hotels only.


In [ ]:
def split_by_year(ano):
    tr=PARES[PARES.f_year.lt(ano)].copy()
    te=PARES[PARES.f_year.eq(ano)].copy()
    if len(tr)<80 or len(te)<30:
        raise ValueError(f'Insufficient sample size for {ano}: training={len(tr)}, test={len(te)}')
    assert tr.f_year.max()<te.f_year.min()
    return tr,te

class ClustersPorTreino:
    def __init__(self,k=5,seed=42):
        self.k=k; self.seed=seed
    def fit(self,df):
        hoteis=df.drop_duplicates('hotel_key',keep='last')
        vals=hoteis[ESP_COLS].replace([np.inf,-np.inf],np.nan)
        self.imp=SimpleImputer(strategy='median')
        arr=self.imp.fit_transform(vals)
        self.q01=np.quantile(arr,.01,axis=0)
        self.q99=np.quantile(arr,.99,axis=0)
        self.scale=StandardScaler().fit(np.clip(arr,self.q01,self.q99))
        Z=self.scale.transform(np.clip(arr,self.q01,self.q99))
        if len(hoteis)<self.k: raise ValueError('Too few hotels for K-Means')
        self.kmeans=KMeans(n_clusters=self.k,n_init=15,random_state=self.seed).fit(Z)
        self.n_hoteis_treino=len(hoteis)
        return self
    def transform(self,df):
        arr=self.imp.transform(df[ESP_COLS].replace([np.inf,-np.inf],np.nan))
        Z=self.scale.transform(np.clip(arr,self.q01,self.q99))
        return pd.Series(['C'+str(i+1) for i in self.kmeans.predict(Z)],index=df.index)

FALHAS=[]
def record_failure(ano,modelo,cenario,erro,etapa='validacao'):
    FALHAS.append({'ano':ano,'modelo':modelo,'cenario':cenario,'etapa':etapa,
                   'tipo':type(erro).__name__,'motivo':str(erro)[:500]})
    print('Not executed:',etapa,ano,modelo,cenario,'-',str(erro)[:135])

def prepare_training_matrices(tr,te,cenario,clusters):
    numericas=FEATS_A if cenario=='A_estrutura_tempo' else FEATS_B
    t1=tr[numericas].replace([np.inf,-np.inf],np.nan).apply(pd.to_numeric,errors='coerce')
    t2=te[numericas].replace([np.inf,-np.inf],np.nan).apply(pd.to_numeric,errors='coerce')
    imp=SimpleImputer(strategy='median')
    Xtr=pd.DataFrame(imp.fit_transform(t1),columns=numericas,index=tr.index)
    Xte=pd.DataFrame(imp.transform(t2),columns=numericas,index=te.index)
    reg1=pd.get_dummies(tr.regiao_lag1, prefix='regiao',dtype=float)
    reg2=pd.get_dummies(te.regiao_lag1, prefix='regiao',dtype=float)
    Xtr=pd.concat([Xtr,reg1],axis=1)
    Xte=pd.concat([Xte,reg2],axis=1)
    if cenario=='C_mais_cluster':
        for df_,tag in [(tr,'tr'),(te,'te')]:
            if 'cluster_treino' not in df_: raise AssertionError('Clusters could not be created from the training data')
        Xtr=pd.concat([Xtr,pd.get_dummies(tr.cluster_treino,prefix='cluster',dtype=float)],axis=1)
        Xte=pd.concat([Xte,pd.get_dummies(te.cluster_treino,prefix='cluster',dtype=float)],axis=1)
    Xte=Xte.reindex(columns=Xtr.columns,fill_value=0.0)
    return Xtr.astype(float),Xte.astype(float)

def build_model(nome):
    if nome=='ridge': return Ridge(alpha=20.0)
    if nome=='random_forest':
        return RandomForestRegressor(n_estimators=160,min_samples_leaf=3,
                                     n_jobs=THREADS,random_state=SEMENTE)
    if nome=='xgboost':
        from xgboost import XGBRegressor
        return XGBRegressor(n_estimators=220,max_depth=4,learning_rate=.05,
                            subsample=.9,colsample_bytree=.9,tree_method='hist',
                            objective='reg:squarederror',n_jobs=THREADS,random_state=SEMENTE)
    if nome=='lightgbm':
        from lightgbm import LGBMRegressor
        return LGBMRegressor(n_estimators=220,learning_rate=.05,max_depth=5,num_leaves=18,
                             min_child_samples=18,verbosity=-1,n_jobs=THREADS,random_state=SEMENTE)
    if nome=='catboost':
        from catboost import CatBoostRegressor
        return CatBoostRegressor(iterations=240,depth=5,learning_rate=.05,
                                 verbose=False,allow_writing_files=False,
                                 thread_count=THREADS,random_seed=SEMENTE)
    if nome in ('tabm','realmlp'):
        from pytabkit import TabM_D_Regressor, RealMLP_TD_Regressor
        device='cpu'
        try:
            import torch
            device='cuda' if torch.cuda.is_available() else 'cpu'
        except ImportError: pass
        klass=TabM_D_Regressor if nome=='tabm' else RealMLP_TD_Regressor
        return klass(device=device,random_state=SEMENTE)
    raise ValueError('Undefined model: '+nome)

def compute_metrics(y,pred):
    y=np.asarray(y,float); pred=np.asarray(pred,float)
    return {'MAE':float(mean_absolute_error(y,pred)),
            'RMSE':float(np.sqrt(mean_squared_error(y,pred))),
            'R2':float(r2_score(y,pred))}

def estimate_predictions(nome,tr,te,cenario):
    Xtr,Xte=prepare_training_matrices(tr,te,cenario,None)
    ytr=tr.f_labprodct.to_numpy(float)
    if POLITICA_Y_TREINO=='p995':
        lim=float(np.quantile(ytr,.995))
        yf=np.clip(ytr,None,lim)
    elif POLITICA_Y_TREINO=='original':
        lim=np.nan; yf=ytr.copy()
    else: raise ValueError('Unknown POLITICA_Y_TREINO')
    modelo=build_model(nome)
    if nome=='ridge':
        scale=StandardScaler()
        train_arr=scale.fit_transform(Xtr)
        test_arr=scale.transform(Xte)
    else:
        train_arr=Xtr.to_numpy(dtype=np.float32)
        test_arr=Xte.to_numpy(dtype=np.float32)
    inicio=time.perf_counter()
    modelo.fit(train_arr,yf)
    pred=np.maximum(np.asarray(modelo.predict(test_arr)).reshape(-1),0.)
    return pred, {'tempo_s':round(time.perf_counter()-inicio,3),
                  'limite_y_treino':lim,'n_features':Xtr.shape[1]}

print('Clustering and training functions are ready.')

## 7. A/B/C comparison for 2021–2023 (main evaluation)

Each test point represents **the same hotel and year** in A, B, and C. Errors are measured on the original scale, without changing observed validation values. Optional models that are not installed or fail to run correctly are logged in `status_modelos.csv`.


In [ ]:
RESULTADOS=[]
PREVISOES=[]
STATUS=[]
METADADOS_FOLDS=[]
for ano in ANOS_CV:
    tr,te=split_by_year(ano)
    grp=ClustersPorTreino(K_CLUSTERS,SEMENTE).fit(tr)
    tr=tr.assign(cluster_treino=grp.transform(tr))
    te=te.assign(cluster_treino=grp.transform(te))
    METADADOS_FOLDS.append({'ano':ano,'treino_n':len(tr),'teste_n':len(te),
                            'hoteis_unicos_cluster_treino':grp.n_hoteis_treino,
                            'treino_ultimo_ano':int(tr.f_year.max()),
                            'k_vizinhos_espaciais':K_VIZINHOS})
    print(f'\nYEAR {ano}: training={len(tr)} | testing={len(te)} | hotels used to fit K-Means={grp.n_hoteis_treino}')
    for modelo in MODELOS:
        for scenario in CONFIGURACOES:
            try:
                previsao,extra=estimate_predictions(modelo,tr,te,scenario)
                met=compute_metrics(te.f_labprodct,previsao)
                RESULTADOS.append({'ano':ano,'modelo':modelo,'cenario':scenario,'n':len(te),**met,**extra})
                PREVISOES.append(pd.DataFrame({
                    'hotel_key':te.hotel_key.to_numpy(),'ano':ano,'real':te.f_labprodct.to_numpy(),
                    'previsto':previsao,'modelo':modelo,'cenario':scenario,
                    'cluster_fold':te.cluster_treino.to_numpy()}))
                STATUS.append({'ano':ano,'modelo':modelo,'cenario':scenario,'status':'EXECUTADO','motivo':''})
            except Exception as e:
                record_failure(ano,modelo,scenario,e)
                STATUS.append({'ano':ano,'modelo':modelo,'cenario':scenario,'status':'NAO_EXECUTADO',
                               'motivo':str(e)[:250]})

cv=pd.DataFrame(RESULTADOS)
pred_cv=pd.concat(PREVISOES,ignore_index=True) if PREVISOES else pd.DataFrame()
status=pd.DataFrame(STATUS)
if cv.empty: raise RuntimeError('No models ran. Check status_modelos.csv and installed packages.')
cv.to_csv(OUT/'05_cv_temporal_A_B_C.csv',index=False)
pred_cv.to_csv(OUT/'06_previsoes_cv_pareadas.csv',index=False)
status.to_csv(OUT/'07_status_modelos.csv',index=False)
pd.DataFrame(METADADOS_FOLDS).to_csv(OUT/'08_metadata_folds.csv',index=False)
display(cv[['ano','modelo','cenario','MAE','RMSE','R2','n']].round(3).head(24))
print('\nCompleted runs:',sum(status.status=='EXECUTADO'),'/',len(status))

## 8. Temporal ranking and **paired** incremental gains

To compare B−A or C−B, we measure $\Delta\mathrm{MAE}=\mathrm{MAE}_{\mathrm{new}}-\mathrm{MAE}_{\mathrm{reference}}$ **on the same observations**. Negative results favour the new scenario.

The main model is selected by **mean MAE across 2021–2023** (with equal year weights), not based on 2024. We also report RMSE and $R^2$, because extreme values may change the ranking.


In [ ]:
ranking=(cv.groupby(['modelo','cenario'],as_index=False)
         .agg(anos=('ano','nunique'),MAE_medio=('MAE','mean'),
              RMSE_medio=('RMSE','mean'),R2_medio=('R2','mean'),segundos=('tempo_s','sum')))
ranking=ranking[ranking.anos.eq(len(ANOS_CV))].sort_values(['MAE_medio','RMSE_medio'])
ranking.to_csv(OUT/'09_ranking_temporal.csv',index=False)
print('RANKING (all years completed; lower mean MAE is better):')
display(ranking.round(3))

pares_delta=[]
for modelo,g in cv.groupby('modelo'):
    for ano in ANOS_CV:
        s=g[g.ano.eq(ano)].set_index('cenario')
        if all(k in s.index for k in CONFIGURACOES):
            for origem,destino in [('A_estrutura_tempo','B_mais_espaco'),
                                   ('B_mais_espaco','C_mais_cluster')]:
                pares_delta.append({'modelo':modelo,'ano':ano,'comparacao':destino+' vs '+origem,
                                    'Delta_MAE':float(s.loc[destino,'MAE']-s.loc[origem,'MAE']),
                                    'Delta_RMSE':float(s.loc[destino,'RMSE']-s.loc[origem,'RMSE']),
                                    'Delta_R2':float(s.loc[destino,'R2']-s.loc[origem,'R2'])})
deltas=pd.DataFrame(pares_delta)
deltas.to_csv(OUT/'10_ganhos_anuais_pareados.csv',index=False)
print('Year-by-year gains (negative ΔMAE = improvement):')
display(deltas.round(3).head(24))

## 9. Comparative figures for the paper: improvements by year and scenario


In [ ]:
melhores=ranking['modelo'].drop_duplicates().tolist()[:4]
fig,axs=plt.subplots(1,len(melhores),figsize=(4.2*len(melhores),4.0),sharey=False,squeeze=False)
for ax,modelo in zip(axs.flat,melhores):
    sub=cv[cv.modelo.eq(modelo)]
    for c in CONFIGURACOES:
        s=sub[sub.cenario.eq(c)].sort_values('ano')
        if len(s): ax.plot(s.ano,s.MAE,marker='o',lw=1.8,color=CORES_CENARIOS[c],label=c.split('_')[0])
    ax.set(title=modelo.upper(),xlabel='Forecast year',ylabel='MAE (lower is better)',xticks=ANOS_CV)
    ax.legend(title='Scenario')
fig.suptitle('Spatial benefit over time (same observations each year)',y=1.03)
fig.tight_layout()
fig.savefig(OUT/'figura_03_MAE_anos_cenarios.png',bbox_inches='tight')
plt.show()

if not deltas.empty:
    dg=deltas[deltas.modelo.isin(melhores)].copy()
    fig,ax=plt.subplots(figsize=(10,4.5))
    tipos=dg.comparacao.drop_duplicates().tolist()
    for j,tipo in enumerate(tipos):
        sub=dg[dg.comparacao.eq(tipo)]
        medias=sub.groupby('modelo')['Delta_MAE'].mean().reindex(melhores)
        pos=np.arange(len(melhores))+((j-(len(tipos)-1)/2)*.32)
        ax.bar(pos,medias.values,width=.31,label=tipo,alpha=.88)
    ax.axhline(0,color='black',lw=1)
    ax.set_xticks(np.arange(len(melhores)),melhores)
    ax.set(title='Average effect of the spatial feature block / clusters, 2021–2023',ylabel='ΔMAE (negative = improvement)')
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT/'figura_04_ganho_incremental.png',bbox_inches='tight')
    plt.show()

## 10. Confidence intervals from hotel-level bootstrapping

To preserve dependence between observations of the same hotel across different years, we resample **hotel identifiers**, not individual rows. In each bootstrap replicate, predictions from A, B, and C are compared using the same resampled observations. We estimate percentile-based 95% confidence intervals for $\Delta\mathrm{MAE}$.

**Important:** these intervals reflect uncertainty associated with the observed hotels, **conditional on the available years**; they do not represent uncertainty about future economic shocks. They do not automatically correct for multiple comparisons or post-hoc model selection. An interval spanning zero does not provide clear evidence of improvement.


In [ ]:
def paired_bootstrap(df_a,df_b,n_boot=N_BOOT,seed=SEMENTE):
    a=df_a[['hotel_key','ano','real','previsto']].rename(columns={'previsto':'a'})
    b=df_b[['hotel_key','ano','previsto']].rename(columns={'previsto':'b'})
    j=a.merge(b,on=['hotel_key','ano'],validate='one_to_one')
    if len(j)==0: return None
    ids=j.hotel_key.unique()
    grupos={str(k):v[['real','a','b']].to_numpy(float) for k,v in j.groupby('hotel_key')}
    todos=[grupos[str(k)] for k in ids]
    rng=np.random.default_rng(seed)
    valores=np.empty(n_boot)
    for it in range(n_boot):
        sorteados=rng.integers(0,len(todos),size=len(todos))
        arr=np.concatenate([todos[x] for x in sorteados])
        valores[it]=np.mean(np.abs(arr[:,2]-arr[:,0]))-np.mean(np.abs(arr[:,1]-arr[:,0]))
    estimativa=np.mean(np.abs(j.b-j.real))-np.mean(np.abs(j.a-j.real))
    return {'n_observacoes':len(j),'n_hoteis':len(ids),'Delta_MAE':estimativa,
            'IC95_inf':np.quantile(valores,.025),'IC95_sup':np.quantile(valores,.975)}

BOOT=[]
for modelo in ranking.modelo.drop_duplicates().tolist():
    for origem,destino in [('A_estrutura_tempo','B_mais_espaco'),
                           ('B_mais_espaco','C_mais_cluster')]:
        a=pred_cv[(pred_cv.modelo==modelo)&(pred_cv.cenario==origem)]
        b=pred_cv[(pred_cv.modelo==modelo)&(pred_cv.cenario==destino)]
        if set(ANOS_CV).issubset(a.ano.unique()) and set(ANOS_CV).issubset(b.ano.unique()):
            res=paired_bootstrap(a,b)
            if res: BOOT.append({'modelo':modelo,'comparacao':destino+' vs '+origem,**res})
boot=pd.DataFrame(BOOT)
boot.to_csv(OUT/'11_bootstrap_hoteis_IC95.csv',index=False)
print('95% CI for ΔMAE:')
display(boot.round(3))

## 11. Sensitivity: exclude 2020 from **training**, without changing validation

We repeat B for Ridge and XGBoost after removing training observations whose **target year** is 2020; we do not remove 2020 simply because the year is inconvenient. This test keeps the evaluation hotels and years identical within each comparison and is a sensitivity experiment, not an adjustment based on examining 2024.


In [ ]:
SENS=[]
if EXECUTAR_SENSIBILIDADE_2020:
    for ano in ANOS_CV:
        tr,te=split_by_year(ano)
        for politica in ('todos','sem2020'):
            tr_at=tr if politica=='todos' else tr[tr.f_year.ne(2020)].copy()
            gru=ClustersPorTreino(K_CLUSTERS,SEMENTE).fit(tr_at)
            tr_at=tr_at.assign(cluster_treino=gru.transform(tr_at))
            te_at=te.assign(cluster_treino=gru.transform(te))
            for model in MODELOS_SENS:
                if model not in MODELOS: continue
                try:
                    p,extra=estimate_predictions(model,tr_at,te_at,'B_mais_espaco')
                    SENS.append({'ano':ano,'modelo':model,'politica':politica,
                                 'n_treino':len(tr_at),**compute_metrics(te_at.f_labprodct,p)})
                except Exception as e:
                    record_failure(ano,model,'sensibilidade_'+politica,e,'sensibilidade')
sens=pd.DataFrame(SENS)
if not sens.empty:
    sens.to_csv(OUT/'12_sensibilidade_ano_2020.csv',index=False)
    display(sens.round(3))
    fig,axs=plt.subplots(1,len(sens.modelo.unique()),figsize=(5.2*len(sens.modelo.unique()),3.8),squeeze=False)
    for ax,model in zip(axs.flat,sens.modelo.unique()):
        su=sens[sens.modelo.eq(model)]
        for pol,c in [('todos','#0072B2'),('sem2020','#D55E00')]:
            t=su[su.politica.eq(pol)].sort_values('ano')
            ax.plot(t.ano,t.MAE,marker='o',label=pol,color=c)
        ax.set(title=model.upper(),xlabel='Evaluation year',ylabel='MAE',xticks=ANOS_CV)
        ax.legend()
    fig.suptitle('Sensitivity to including 2020 observations in training')
    fig.tight_layout()
    fig.savefig(OUT/'figura_05_sensibilidade_2020.png',bbox_inches='tight')
    plt.show()

## 12. Retrospective audit for 2024: **do not use for model selection**

The year 2024 has already been examined in earlier research and therefore **is not an independent blind test**. This block fits models on pairs with targets up to 2023 and checks what happens in 2024 with the previously fixed parameters. The A/B/C comparison retains the same hotel cohort.


In [ ]:
AUDIT=[]; PREV24=[]
if EXECUTAR_AUDITORIA_2024:
    tr,te=split_by_year(ANO_AUDITORIA)
    gr=ClustersPorTreino(K_CLUSTERS,SEMENTE).fit(tr)
    tr=tr.assign(cluster_treino=gr.transform(tr))
    te=te.assign(cluster_treino=gr.transform(te))
    for model in MODELOS:
        if model not in set(ranking.modelo): continue
        for c in CONFIGURACOES:
            try:
                p,extra=estimate_predictions(model,tr,te,c)
                AUDIT.append({'ano':2024,'modelo':model,'cenario':c,'n':len(te),
                              **compute_metrics(te.f_labprodct,p),**extra})
                PREV24.append(pd.DataFrame({'hotel_key':te.hotel_key.to_numpy(),
                    'ano':2024,'real':te.f_labprodct.to_numpy(),'previsto':p,
                    'modelo':model,'cenario':c}))
            except Exception as e:
                record_failure(2024,model,c,e,'auditoria_2024')
audit24=pd.DataFrame(AUDIT)
pred24=pd.concat(PREV24,ignore_index=True) if PREV24 else pd.DataFrame()
if not audit24.empty:
    audit24.to_csv(OUT/'13_auditoria_2024.csv',index=False)
    pred24.to_csv(OUT/'14_previsoes_2024.csv',index=False)
    print('RETROSPECTIVE 2024 AUDIT (not independent):')
    display(audit24.sort_values('MAE')[['modelo','cenario','MAE','RMSE','R2','n']].round(3))

## 13. Variables associated with changes in spatial profiles — regional-composition diagnostic


In [ ]:
# Cross-tabulate profiles and NUTS II regions to help identify regional confounding.
# This does not mean that a cluster causes or changes productivity.
if not descritivo.empty:
    profis=espaco[['hotel_key','cluster']].drop_duplicates()
    reg=base.sort_values('f_year').drop_duplicates('hotel_key',keep='last')[['hotel_key','ht_nuts_2_newcriteria']]
    regs=reg.merge(profis,on='hotel_key',validate='one_to_one')
    tab=pd.crosstab(regs.ht_nuts_2_newcriteria.fillna('Region unspecified'),regs.cluster)
    tab.to_csv(OUT/'15_regiao_por_cluster.csv')
    top=tab.loc[tab.sum(axis=1).nlargest(9).index]
    fig,ax=plt.subplots(figsize=(10.5,5))
    mat=top.div(top.sum(axis=1),axis=0)
    mat.plot(kind='barh',stacked=True,ax=ax,
             color=[CORES_PERFIS.get(int(x),'#777') for x in mat.columns])
    ax.set(xlabel='Share of hotels in region',ylabel='NUTS II region',
           title='Distribution of spatial profiles by region')
    ax.legend(title='Cluster',bbox_to_anchor=(1.02,1),loc='upper left')
    fig.tight_layout()
    fig.savefig(OUT/'figura_06_composicao_regional.png',bbox_inches='tight')
    plt.show()

## 14. Execution report, limitations, and export

**Three important considerations for the manuscript:**

- **Temporal provenance of POIs.** If the POIs date from 2026, evaluation over 2021–2024 measures a retrospective association with later geographic information, not forecasting performance using historically available predictors. Before claiming genuine *forecasting*, reconstruct year-specific spatial attributes or document a defensible stability assumption.
- **Clusters as a deterministic transformation.** Scenario C does not provide additional data beyond B; a benefit is representational and may disappear with more expressive models.
- **Transparent comparison.** Indicators, clustering, imputation, target thresholds, and hyperparameters are chosen without consulting the evaluation-year target. However, the overall study design remains exploratory because 2024 has already been examined; results should be replicated using another horizon or dataset.

**Candidate figures for KBS:** annual evolution by profile, paired changes from 2019 to 2020, A/B/C performance by year, incremental benefit with confidence intervals, and regional composition.


In [ ]:
metadata={
 'projeto':'KBS - Luis Silvano - Notebook 06',
 'data_ref_poi':'Unknown; confirm with author',
 'fonte_painel':ARQ_PAINEL.name,
 'fonte_espacial':str(ARQ_ESPACO.relative_to(ROOT)),
 'k_vizinhos':K_VIZINHOS,'k_clusters_por_treino':K_CLUSTERS,
 'modelos_solicitados':MODELOS,'anos_cv':ANOS_CV,'auditoria_2024':EXECUTAR_AUDITORIA_2024,
 'politica_y_treino':POLITICA_Y_TREINO,'n_obs_pares':len(PARES),
 'n_hoteis_pares':PARES.hotel_key.nunique(),
 'definicao_A':FEATS_A+['regiao_lag1'],
 'definicao_B':FEATS_B+['regiao_lag1'],
 'definicao_C':FEATS_B+['regiao_lag1','cluster_treino'],
 'interpretação_C':'Cluster is a deterministic transformation of features in B.',
 'avisos':['2024 was inspected previously — not a blind test.',
           'POIs may postdate the productivity period.',
           'No causal inference.'],
 'versoes':{'python':sys.version.split()[0],'numpy':np.__version__,
            'pandas':pd.__version__}
}
(OUT/'16_metadados_estudo.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
if FALHAS: pd.DataFrame(FALHAS).to_csv(OUT/'17_falhas_modelos.csv',index=False)
print('MODELS / STATUS:')
display(status.groupby(['modelo','status']).size().unstack(fill_value=0))
if not ranking.empty:
    vencedor=ranking.iloc[0]
    print('Selection based exclusively on mean MAE for 2021–2023:',
          vencedor.modelo,'|',vencedor.cenario,'| MAE=',round(vencedor.MAE_medio,3))
print('\nFicheiros gerados:')
for arquivo in sorted(OUT.iterdir()):
    if arquivo.is_file(): print(' -',arquivo.name)

## 15. Draft text for the experimental section (edit after confirming results)

> *We assessed the incremental predictive contribution of spatial accessibility indicators and tourism-accessibility profiles using a rolling-origin evaluation on an annual hotel panel. For each validation year, temporal lags and all preprocessing parameters were estimated using previous years only. An unsupervised K-Means model was fitted on unique hotels in the training period, and validation hotel-year observations were assigned to the learned centroids. Three nested feature configurations were examined: structural–temporal controls (A), additional continuous spatial accessibility indicators (B), and spatial profile membership on top of those indicators (C). The outcome was never used in clustering. Comparisons were paired at the hotel–year level, and uncertainty was evaluated by resampling hotel identities. The 2024 evaluation was reported exclusively as a retrospective audit, since it had been inspected during model development.*

**Before submission:** document POI sources and collection dates, check units and coverage, perform a genuinely uninspected external test, report temporal variation and uncertainty, justify selection of the independent-hotel sample, and assess sensitivity to extreme values.

### Interpreting the results

- **B consistently outperforms A:** spatial indicators provide additional predictive value for the examined models and periods (not necessarily causal).
- **C consistently outperforms B:** spatial discretisation through clustering may make the predictive function easier to learn; **this does not prove that new information is available**.
- **B does not outperform A:** location may already be partly captured by region, hotel capacity, and history, or the POIs may be temporally misaligned.
- **Gains differ between 2021 and 2023:** heterogeneous economic regimes, pandemic-related shocks, and extreme values deserve investigation, rather than opportunistic selection of a single year.
